# E-commerce Demand Forecasting — Weekly Category Demand

Holt-Winters exponential smoothing on synthetic weekly demand by product
category, with an 8-week holdout to measure real forecast accuracy (MAPE)
before producing a forward forecast. See `docs/methodology.md` for why this
uses trend-only smoothing rather than a full seasonal fit.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing

demand = pd.read_csv("../data/weekly_demand_by_category.csv", parse_dates=["week"])
print(demand.shape)
print(demand["category"].unique())
demand.head()


(416, 3)
<StringArray>
[     'Suspension',  'Braking System',    'Engine Parts',         'Filters',
        'Lighting', 'Body & Exterior',  'Cooling System',    'Transmission']
Length: 8, dtype: str


## 1. Look at one category in detail: Suspension

In [2]:
s = demand[demand["category"]=="Suspension"].sort_values("week").set_index("week")["units_sold"]
print(f"{len(s)} weeks, mean {s.mean():.0f} units/week, min {s.min()}, max {s.max()}")
s.describe()


52 weeks, mean 91 units/week, min 55, max 128


## 2. Train/holdout split and Holt-Winters fit

In [3]:
def mape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    mask = actual != 0
    return float(np.mean(np.abs((actual[mask]-forecast[mask])/actual[mask]))*100)

HOLDOUT = 8
train, test = s.iloc[:-HOLDOUT], s.iloc[-HOLDOUT:]

model = ExponentialSmoothing(train, trend="add", initialization_method="estimated").fit()
holdout_forecast = model.forecast(HOLDOUT)

print("Holdout actual:  ", test.values)
print("Holdout forecast:", holdout_forecast.round(0).values)
print(f"MAPE on holdout: {mape(test.values, holdout_forecast.values):.1f}%")


Holdout actual:   [99 95 92 76 73 89 89 64]
Holdout forecast: [97. 94. 91. 88. 85. 82. 79. 76.]
MAPE on holdout: 9.2%


## 3. Run the same fit across every category

In [4]:
results = []
for category, g in demand.groupby("category"):
    series = g.sort_values("week").set_index("week")["units_sold"]
    train, test = series.iloc[:-HOLDOUT], series.iloc[-HOLDOUT:]
    m = ExponentialSmoothing(train, trend="add", initialization_method="estimated").fit()
    fc = m.forecast(HOLDOUT)
    results.append({"category": category, "mape_pct": round(mape(test.values, fc.values), 1),
                     "avg_weekly_units": round(series.mean(), 0)})

results_df = pd.DataFrame(results).sort_values("mape_pct")
results_df


## 4. Key findings

1. Forecast accuracy varies a lot by category: MAPE from ~9% (Transmission,
   Suspension — high volume, low noise) to ~40% (Engine Parts, Lighting —
   lower volume, noisier week to week). Averaging accuracy across categories
   into one number would hide this and lead to one safety-stock policy
   applied everywhere, which is wrong in both directions.
2. One year of history is not enough to fit a reliable seasonal component —
   the model here is trend-only, and that's a deliberate, documented choice,
   not an oversight (see `docs/methodology.md`).
3. Categories with high MAPE are exactly the ones that need a bigger safety
   stock buffer, not the ones where a tighter forecast would help most —
   the forecast quality itself is operationally useful information, not
   just the forecast value.